This combines the results of Ramya and Marcus' final re-labeling with the old labels

In [56]:
import json
import pandas as pd
import ast
import numpy as np

In [42]:
old_df = pd.read_csv('/data/vision/polina/users/marcusbl/bin_class/label_sessions_data/label_session_3-11/post_session.csv', index_col=0)
old_df['label_3'] = None

In [43]:
old_df.head()

,slice_num,stack_slices_cnt,full_stack_slices_cnt,stack_slices,path,Brain Type,person,dataset,mask_path,MAP ID,GA,person_id,label_R,label_1,is_edge,label_2,labels_agree,req_relabel,tie_broken,label_3
0,1,10,17,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",/data/vision/polina/users/marcusbl/data/anon-0...,NaN,anon-00003,R,/data/vision/polina/users/marcusbl/data/anon-0...,NaN,NaN,27,NaN,NaN,NaN,NaN,False,NaN,False,None
1,2,10,17,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",/data/vision/polina/users/marcusbl/data/anon-0...,NaN,anon-00003,R,/data/vision/polina/users/marcusbl/data/anon-0...,NaN,NaN,27,1.0,NaN,NaN,NaN,False,NaN,False,None
2,3,10,17,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",/data/vision/polina/users/marcusbl/data/anon-0...,NaN,anon-00003,R,/data/vision/polina/users/marcusbl/data/anon-0...,NaN,NaN,27,0.0,NaN,NaN,NaN,False,NaN,False,None
3,4,10,17,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",/data/vision/polina/users/marcusbl/data/anon-0...,NaN,anon-00003,R,/data/vision/polina/users/marcusbl/data/anon-0...,NaN,NaN,27,1.0,NaN,NaN,NaN,False,NaN,False,None
4,5,10,17,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]",/data/vision/polina/users/marcusbl/data/anon-0...,NaN,anon-00003,R,/data/vision/polina/users/marcusbl/data/anon-0...,NaN,NaN,27,0.0,NaN,NaN,NaN,False,NaN,False,None


In [44]:
labels3_df = pd.read_csv('/data/vision/polina/users/marcusbl/bin_class/label_sessions_data/label_session_3-11/post_label_session/Labels_MR - Sheet1.csv', 
                         index_col='index',
                         header = 1).iloc[:, 1:]
labels3_df.head()

,M_label,R_label (inverse),Different?,R_label
index,,,,
0,1,0.0,0,1
1,1,0.0,0,1
2,0,0.0,0,0
3,0,0.0,0,0
4,0,1.0,0,0


In [45]:
lookup = {}
with open('/data/vision/polina/users/marcusbl/bin_class/label_sessions_data/label_session_3-11/post_label_session/lookup.json', 'r') as f:
    lookup = json.load(f)


In [46]:
for key_str, (path, slice_num) in lookup.items():
    # Parse key string
    dash_idx = str.index(key_str, '-')

    # Get label from the df
    df_idx = int(key_str[dash_idx+1:])
    label = labels3_df.loc[df_idx]['M_label']

    # Update the original df at label_3 column
    old_df_filter = (old_df['path'] == str(path)) & (old_df['slice_num'] == slice_num)
    if label == '-':
        old_df.loc[old_df_filter, 'is_edge'] = True
    else:
        old_df.loc[old_df_filter, 'is_edge'] = False
        old_df.loc[old_df_filter, 'label_3'] = label

/tmp/ipykernel_50687/2657312443.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  old_df.loc[old_df_filter, 'is_edge'] = False


In [47]:
old_df['label_3'].value_counts()

label_3
0    845
1    311
Name: count, dtype: int64

In [48]:
print(labels3_df['M_label'].value_counts())

M_label
0    845
1    311
-     28
Name: count, dtype: int64


Final Label: 
- If label_3, chooose that
- Otherwise, choose majority of label_2

In [66]:
for idx, row in old_df.iterrows():
    if row['is_edge']:
        continue

    if pd.notna(row['label_3']):
        old_df.at[idx, 'final_label'] = int(row['label_3'])
    else:
        if isinstance(row['label_2'], float):  # NaN
            continue

        label_2 = ast.literal_eval(row['label_2'])
        assert label_2[0] == label_2[1]

        old_df.at[idx, 'label_3'] = int(label_2[0])
        old_df.at[idx, 'final_label'] = int(label_2[0])

In [71]:
old_df['final_label'].value_counts()

final_label
0    5781
1    1189
Name: count, dtype: int64

In [72]:
old_df['is_edge'].value_counts()

is_edge
False    6970
1.0      1426
Name: count, dtype: int64

In [73]:
old_df.to_csv('/data/vision/polina/users/marcusbl/bin_class/label_sessions_data/label_session_3-11/final.csv')